# MNIST Rotation Dataset — Creation & Analysis

Creates three augmented MNIST rotation datasets and analyses the augmentation counts.

**Three-stage pipeline (thesis Section 4.2.1):**
1. Train a CNN classifier on original MNIST (>99% accuracy)
2. Test candidate rotation angles; accept rotated images where classifier confidence > 0.9999
3. Apply a final uniform random rotation θ ~ Uniform(0°, 360°) to every entry

**Three dataset variants:**
| Name | Candidate angles | Balanced option |
|------|-----------------|----------------|
| `thesis`  | 90°, 180°, 270° | ✓ |
| `every45` | 45°, 90°, 135°, 180°, 225°, 270°, 315° | ✓ |
| `every10` | 10°, 20°, ..., 350° | ✓ |

**Balanced sampling:** for each digit, resample the augmented entries so the total count per digit equals the original MNIST count. This prevents the model from being biased toward digits that received many augmented copies (e.g. digit '1' at 180° → 6,053 extra entries).

---
## 0. Imports & Setup

In [ ]:
import math, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = './data'
OUTPUT_DIR = './data/mnist_rotation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.9999
CLASSIFIER_MEAN = 0.1307
CLASSIFIER_STD  = 0.3081

print(f'Device: {device}')
print(f'Output dir: {OUTPUT_DIR}')

---
## 1. Load Raw MNIST

In [ ]:
raw_ds = torchvision.datasets.MNIST(DATA_DIR, train=True, download=True,
                                     transform=transforms.ToTensor())
raw_loader = DataLoader(raw_ds, batch_size=4096, shuffle=False, num_workers=2)

all_imgs, all_lbls = [], []
for imgs, lbls in tqdm(raw_loader, desc='Loading MNIST'):
    all_imgs.append(imgs)
    all_lbls.append(lbls)

raw_images = torch.cat(all_imgs, dim=0)   # [60000, 1, 28, 28]  in [0, 1]
raw_labels = torch.cat(all_lbls, dim=0)   # [60000]

# Original count per digit — used later for balanced sampling
original_counts = {d: (raw_labels == d).sum().item() for d in range(10)}

print(f'Loaded {raw_images.shape[0]:,} images')
print('Label distribution (original MNIST):')
for d in range(10):
    print(f'  Digit {d}: {original_counts[d]:,}')

---
## 2. Stage 1 — Train CNN Classifier

Architecture: three conv blocks + FC layers (thesis Appendix 6.2.1).  
Hyperparameters: AdamW lr=1e-3, ReduceLROnPlateau(factor=0.5, patience=2), 15 epochs.

In [ ]:
class MNISTClassifier(nn.Module):
    """CNN classifier — thesis Appendix 6.2.1 / Figure 6.9."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.25),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*7*7, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.block3(self.block2(self.block1(x))))

In [ ]:
CLASSIFIER_PATH    = os.path.join(OUTPUT_DIR, 'mnist_classifier.pt')
NEPOCHS_CLASSIFIER = 15

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([CLASSIFIER_MEAN], [CLASSIFIER_STD]),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([CLASSIFIER_MEAN], [CLASSIFIER_STD]),
])

train_cls_ds     = torchvision.datasets.MNIST(DATA_DIR, train=True,  transform=train_transform)
test_cls_ds      = torchvision.datasets.MNIST(DATA_DIR, train=False, transform=test_transform)
train_cls_loader = DataLoader(train_cls_ds, batch_size=128, shuffle=True,  num_workers=2)
test_cls_loader  = DataLoader(test_cls_ds,  batch_size=256, shuffle=False, num_workers=2)

if os.path.exists(CLASSIFIER_PATH):
    print(f'Loading existing classifier from {CLASSIFIER_PATH}')
    classifier = MNISTClassifier().to(device)
    classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location=device))
    classifier.eval()
else:
    classifier = MNISTClassifier().to(device)
    optimizer  = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched      = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    criterion  = nn.CrossEntropyLoss()

    for epoch in range(1, NEPOCHS_CLASSIFIER + 1):
        classifier.train()
        for imgs, lbls in tqdm(train_cls_loader, desc=f'Epoch {epoch}/{NEPOCHS_CLASSIFIER}', leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            criterion(classifier(imgs), lbls).backward()
            optimizer.step()

        classifier.eval()
        correct = total = val_loss = 0
        with torch.no_grad():
            for imgs, lbls in test_cls_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = classifier(imgs)
                val_loss += criterion(logits, lbls).item()
                correct  += (logits.argmax(1) == lbls).sum().item()
                total    += lbls.size(0)
        acc = correct / total
        sched.step(val_loss)
        print(f'  Epoch {epoch:2d} | val_acc={acc:.4f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

    torch.save(classifier.state_dict(), CLASSIFIER_PATH)
    print(f'Classifier saved to {CLASSIFIER_PATH}')
    classifier.eval()

In [ ]:
classifier.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in test_cls_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        correct += (classifier(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
print(f'Test accuracy: {correct/total*100:.2f}%')

---
## 3. Core Pipeline Functions

In [ ]:
@torch.no_grad()
def run_stage2(raw_images, raw_labels, classifier, candidate_angles_deg,
               confidence_threshold=0.9999, batch_size=512):
    """
    Stage 2: Logical Angle Assignment (thesis Section 4.2.1).

    For each original image, rotate by each candidate angle.
    Accept if classifier predicts any digit with confidence >= threshold.
    Originals are always included with base_angle = 0.

    Returns:
        aug_images      [M, 1, 28, 28]  float in [0, 1]
        aug_labels      [M]             predicted digit label
        aug_base_angles [M]             discrete rotation angle (0 = original)
        stats           dict  angle -> per-digit counts  (for analysis)
    """
    classifier.eval()
    N = raw_images.shape[0]

    all_imgs = [raw_images]
    all_lbls = [raw_labels]
    all_base = [torch.zeros(N, dtype=torch.float32)]
    stats    = {a: [0]*10 for a in candidate_angles_deg}

    for angle_deg in tqdm(candidate_angles_deg, desc='Stage 2: scanning angles'):
        a_imgs, a_lbls, a_base = [], [], []
        for s in range(0, N, batch_size):
            e            = min(s + batch_size, N)
            imgs         = raw_images[s:e].to(device)
            rot          = TF.rotate(imgs, angle=float(angle_deg))
            rot_norm     = (rot - CLASSIFIER_MEAN) / CLASSIFIER_STD
            probs, preds = F.softmax(classifier(rot_norm), dim=1).max(dim=1)
            mask         = probs >= confidence_threshold
            if mask.sum() > 0:
                accepted = rot[mask].cpu()
                labels   = preds[mask].cpu()
                a_imgs.append(accepted)
                a_lbls.append(labels)
                a_base.append(torch.full((accepted.shape[0],), float(angle_deg)))
                for lbl in labels:
                    stats[angle_deg][lbl.item()] += 1
        if a_imgs:
            all_imgs.append(torch.cat(a_imgs))
            all_lbls.append(torch.cat(a_lbls))
            all_base.append(torch.cat(a_base))

    return (torch.cat(all_imgs), torch.cat(all_lbls),
            torch.cat(all_base), stats)


@torch.no_grad()
def run_stage3(images, base_angles, batch_size=512):
    """
    Stage 3: Uniform Rotation (thesis Section 4.2.1).
    θ ~ Uniform(0°, 360°) per entry.  Final angle = (base_angle + θ) mod 360.
    """
    N              = images.shape[0]
    uniform_angles = torch.rand(N) * 360.0
    final_angles   = (base_angles + uniform_angles) % 360.0
    rotated        = torch.empty_like(images)
    for s in tqdm(range(0, N, batch_size), desc='Stage 3: uniform rotation'):
        e = min(s + batch_size, N)
        for i, (img, ang) in enumerate(zip(images[s:e], uniform_angles[s:e])):
            rotated[s+i] = TF.rotate(img.unsqueeze(0), float(ang.item())).squeeze(0)
    return rotated, final_angles


def balance_dataset(images_01, labels, base_angles, original_counts):
    """
    Balanced sampling.

    For each digit d:
      - Keep ALL original entries  (base_angle == 0).
      - Sample uniformly from augmented entries (base_angle > 0) until
        total count for digit d equals original_counts[d].

    If a digit received fewer augmented entries than needed, all augmented
    entries are kept and the total may fall slightly below original_counts[d].
    (No oversampling with replacement.)

    This ensures the model sees each digit equally often, without being
    dominated by digits that pass the rotation test at many angles.
    """
    keep_indices = []
    for d in range(10):
        target = original_counts[d]

        orig_idx = torch.where((labels == d) & (base_angles == 0))[0]
        aug_idx  = torch.where((labels == d) & (base_angles > 0))[0]
        keep_indices.append(orig_idx)

        n_needed = target - orig_idx.shape[0]
        n_sample = min(n_needed, aug_idx.shape[0])
        if n_sample > 0:
            perm = torch.randperm(aug_idx.shape[0])[:n_sample]
            keep_indices.append(aug_idx[perm])

    keep = torch.cat(keep_indices)
    return images_01[keep], labels[keep], base_angles[keep]


def build_dataset_dict(images_01, labels, base_angles, final_angles, meta):
    """Package arrays into a training-ready dict."""
    imgs_norm  = images_01 * 2.0 - 1.0         # [0,1] → [-1,1]
    angles_rad = final_angles * (math.pi / 180.0)
    return {
        'images':      imgs_norm,
        'images_flat': imgs_norm.view(-1, 784),
        'angles_deg':  final_angles,
        'angle_vec':   torch.stack([torch.cos(angles_rad),
                                    torch.sin(angles_rad)], dim=1),  # (cos, sin)
        'labels':      labels,
        'base_angles': base_angles,
        'meta':        meta,
    }


def make_and_save(name, candidate_angles, raw_images, raw_labels,
                  original_counts, classifier):
    """
    Full pipeline for one dataset variant.
    Saves:  <name>.pt              (full augmented)
            <name>_balanced.pt     (per-digit balanced)
    Returns (data_full, data_balanced, stats).
    """
    path_full = os.path.join(OUTPUT_DIR, f'mnist_rotation_{name}.pt')
    path_bal  = os.path.join(OUTPUT_DIR, f'mnist_rotation_{name}_balanced.pt')

    # ── Full dataset ──────────────────────────────────────────
    if os.path.exists(path_full):
        print(f'[{name}] Loading full dataset from disk...')
        data_full = torch.load(path_full, map_location='cpu')
        stats = {a: [(data_full['labels'][
                          data_full['base_angles'] == float(a)] == d).sum().item()
                     for d in range(10)]
                 for a in candidate_angles}
    else:
        print(f'\n{"="*60}\nBuilding [{name}] | angles: {candidate_angles}\n{"="*60}')
        aug_imgs, aug_lbls, aug_base, stats = run_stage2(
            raw_images, raw_labels, classifier,
            candidate_angles_deg=candidate_angles,
            confidence_threshold=CONFIDENCE_THRESHOLD,
        )
        final_imgs, final_angles = run_stage3(aug_imgs, aug_base)
        N = final_imgs.shape[0]
        data_full = build_dataset_dict(
            final_imgs, aug_lbls, aug_base, final_angles,
            meta={'name': name, 'candidate_angles': candidate_angles,
                  'balanced': False, 'n_total': N,
                  'n_augmented': N - raw_images.shape[0]}
        )
        torch.save(data_full, path_full)
        print(f'  Saved {path_full}  ({N:,} entries)')

    # ── Balanced dataset ─────────────────────────────────────
    if os.path.exists(path_bal):
        print(f'[{name}_balanced] Loading from disk...')
        data_balanced = torch.load(path_bal, map_location='cpu')
    else:
        print(f'  Building balanced version...')
        # Work in [0,1] space for TF.rotate, then normalise at the end
        imgs_01 = (data_full['images'] + 1.0) / 2.0   # back to [0,1]
        bal_imgs, bal_lbls, bal_base = balance_dataset(
            imgs_01, data_full['labels'], data_full['base_angles'], original_counts
        )
        bal_final_imgs, bal_final_angles = run_stage3(bal_imgs, bal_base)
        N_bal = bal_final_imgs.shape[0]
        data_balanced = build_dataset_dict(
            bal_final_imgs, bal_lbls, bal_base, bal_final_angles,
            meta={'name': f'{name}_balanced', 'candidate_angles': candidate_angles,
                  'balanced': True, 'n_total': N_bal,
                  'n_augmented': N_bal - sum(original_counts.values())}
        )
        torch.save(data_balanced, path_bal)
        print(f'  Saved {path_bal}  ({N_bal:,} entries)')

    return data_full, data_balanced, stats

---
## 4. Build All Three Datasets

In [ ]:
THESIS_ANGLES  = [90, 180, 270]
EVERY45_ANGLES = [45, 90, 135, 180, 225, 270, 315]
EVERY10_ANGLES = list(range(10, 360, 10))   # 35 angles

thesis_data,  thesis_balanced,  thesis_stats  = make_and_save(
    'thesis',  THESIS_ANGLES,  raw_images, raw_labels, original_counts, classifier)

every45_data, every45_balanced, every45_stats = make_and_save(
    'every45', EVERY45_ANGLES, raw_images, raw_labels, original_counts, classifier)

every10_data, every10_balanced, every10_stats = make_and_save(
    'every10', EVERY10_ANGLES, raw_images, raw_labels, original_counts, classifier)

---
## 5. Heatmap — (digit × angle)

In [ ]:
heat_data = np.array([thesis_stats[a] for a in THESIS_ANGLES])   # [35, 10]
heat_df   = pd.DataFrame(heat_data,
                          index=[f'{a}°' for a in THESIS_ANGLES],
                          columns=[str(d) for d in range(10)])

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(heat_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Number of augmented images'})
ax.set_xlabel('Digit', fontsize=13)
ax.set_ylabel('Rotation angle', fontsize=13)
ax.set_title('Augmented images per digit per rotation angle\n'
             f'(confidence > {CONFIDENCE_THRESHOLD}, every-90° variant)',
             fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
heat_data = np.array([every45_stats[a] for a in EVERY45_ANGLES])   # [35, 10]
heat_df   = pd.DataFrame(heat_data,
                          index=[f'{a}°' for a in EVERY45_ANGLES],
                          columns=[str(d) for d in range(10)])

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(heat_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Number of augmented images'})
ax.set_xlabel('Digit', fontsize=13)
ax.set_ylabel('Rotation angle', fontsize=13)
ax.set_title('Augmented images per digit per rotation angle\n'
             f'(confidence > {CONFIDENCE_THRESHOLD}, every-45° variant)',
             fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
heat_data = np.array([every10_stats[a] for a in EVERY10_ANGLES])   # [35, 10]
heat_df   = pd.DataFrame(heat_data,
                          index=[f'{a}°' for a in EVERY10_ANGLES],
                          columns=[str(d) for d in range(10)])

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(heat_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Number of augmented images'})
ax.set_xlabel('Digit', fontsize=13)
ax.set_ylabel('Rotation angle', fontsize=13)
ax.set_title('Augmented images per digit per rotation angle\n'
             f'(confidence > {CONFIDENCE_THRESHOLD}, every-10° variant)',
             fontsize=14, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'heatmap_every10.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Balanced Sampling

For each digit `d`, sample exactly `original_counts[d]` entries **uniformly at random**
from the full pool (originals + augmented combined).  
Result: same total size as original MNIST (60,000), but each entry may be a
rotated copy — so the model sees diverse orientations without any digit being over-represented.

In [ ]:
def balance_dataset(data_full, original_counts):
    """
    For each digit d, sample exactly original_counts[d] entries
    uniformly at random from ALL entries for that digit
    (originals + augmented combined).
    """
    labels     = data_full['labels']
    base_angles = data_full['base_angles']
    keep_indices = []

    for d in range(10):
        all_idx  = torch.where(labels == d)[0]
        n_sample = min(original_counts[d], all_idx.shape[0])
        perm     = torch.randperm(all_idx.shape[0])[:n_sample]
        keep_indices.append(all_idx[perm])

    keep = torch.cat(keep_indices)
    return {
        k: v[keep] if isinstance(v, torch.Tensor) and v.shape[0] == labels.shape[0]
        else v
        for k, v in data_full.items()
    }


thesis_balanced  = balance_dataset(thesis_data,  original_counts)
every45_balanced = balance_dataset(every45_data, original_counts)
every10_balanced = balance_dataset(every10_data, original_counts)

# Sanity check
for name, data_b in [('thesis', thesis_balanced),
                      ('every45', every45_balanced),
                      ('every10', every10_balanced)]:
    counts = {d: (data_b['labels'] == d).sum().item() for d in range(10)}
    print(f'{name}_balanced — per-digit counts: {counts}')
    print(f'  total: {data_b["labels"].shape[0]:,}  |  '
          f'augmented fraction: '
          f'{(data_b["base_angles"] > 0).float().mean()*100:.1f}%')

---
## 7. Heatmap — Balanced Datasets

Shows how many of the sampled entries come from each rotation angle.  
Because we sample uniformly from the full pool, the augmented entries
appear roughly in proportion to their size — digits with many rotated
copies (e.g. '0', '1', '8') will still be well-represented across angles.

In [ ]:
def stats_from_dataset(dataset, candidate_angles):
    """Recompute per-angle per-digit counts from base_angles + labels tensors."""
    base = dataset['base_angles']
    lbls = dataset['labels']
    return {
        a: [(lbls[base == float(a)] == d).sum().item() for d in range(10)]
        for a in candidate_angles
    }


def plot_heatmap_pair(stats_full, stats_bal, candidate_angles,
                      variant_name, save_name):
    """Side-by-side heatmap: full augmented (left) vs balanced (right)."""
    def to_df(stats):
        return pd.DataFrame(
            np.array([stats[a] for a in candidate_angles]),
            index=[f'{a}°' for a in candidate_angles],
            columns=[str(d) for d in range(10)]
        )

    fig, axes = plt.subplots(1, 2, figsize=(20, max(4, len(candidate_angles)*0.55)))

    for ax, stats, title in zip(axes,
            [stats_full, stats_bal],
            [f'{variant_name} — full augmented',
             f'{variant_name} — balanced (n = original MNIST per digit)']):
        sns.heatmap(to_df(stats), annot=True, fmt='d', cmap='YlOrRd',
                    linewidths=0.4, ax=ax,
                    cbar_kws={'label': 'Number of images'})
        ax.set_xlabel('Digit', fontsize=12)
        ax.set_ylabel('Rotation angle', fontsize=12)
        ax.set_title(title, fontsize=13, pad=10)

    plt.suptitle(f'Images per digit per angle — {variant_name}\n'
                 f'(confidence > {CONFIDENCE_THRESHOLD})',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150, bbox_inches='tight')
    plt.show()


# ── Thesis (90/180/270) ───────────────────────────────────────
plot_heatmap_pair(
    thesis_stats,
    stats_from_dataset(thesis_balanced, THESIS_ANGLES),
    THESIS_ANGLES, 'Thesis (90/180/270°)', 'heatmap_thesis_balanced.png'
)

# ── Every-45° ─────────────────────────────────────────────────
plot_heatmap_pair(
    every45_stats,
    stats_from_dataset(every45_balanced, EVERY45_ANGLES),
    EVERY45_ANGLES, 'Every-45°', 'heatmap_every45_balanced.png'
)

# ── Every-10° ─────────────────────────────────────────────────
plot_heatmap_pair(
    every10_stats,
    stats_from_dataset(every10_balanced, EVERY10_ANGLES),
    EVERY10_ANGLES, 'Every-10°', 'heatmap_every10_balanced.png'
)